In [4]:

from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, Side

BASE_DIR = Path.cwd()
if not (BASE_DIR / "model_training").exists() and BASE_DIR.name == "model_training":
    BASE_DIR = BASE_DIR.parent

CSV_PATH = BASE_DIR / "model_training" / "dataset_with_condition.csv"
TEMPLATE_PATH = BASE_DIR / "model_training" / "Template_for_Model.xlsx"
OUTPUT_PATH = BASE_DIR / "model_training" / "Template_for_Model_Filled.xlsx"

MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02"]
DATA_START_ROW = 4


def _first_existing_column(df, candidates):
    return next((col for col in candidates if col in df.columns), None)


def _create_fallback_template(template_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Vehicle Prices"
    ws["A1"] = "Vehicle Price Forecast Template (Auto-generated)"

    headers = [
        "Make",
        "Model",
        "Year of Manufacture",
        "NOV 2025",
        "DEC 2025",
        "JAN 2026",
        "FEB 2026",
        "MARCH 2026",
        "APRIL 2026",
        "Next Week Price",
        "AVG. Price | AVG. Mileage",
    ]
    for idx, title in enumerate(headers, start=1):
        cell = ws.cell(row=3, column=idx, value=title)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    wb.save(template_path)
    print(f"Template not found. Created fallback template -> {template_path}")


def load_and_prepare(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)

    required = ["Make", "Model", "Year", "Price", "published date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}")

    df["published date"] = pd.to_datetime(df["published date"], errors="coerce")
    df = df.dropna(subset=["published date"])
    df["month"] = df["published date"].dt.to_period("M").astype(str)
    return df


def build_monthly_pivot(df, months):
    pivot = (
        df.groupby(["Make", "Model", "Year", "month"])["Price"]
        .mean()
        .unstack("month")
    )
    pivot.columns = [str(c) for c in pivot.columns]
    for m in months:
        if m not in pivot.columns:
            pivot[m] = np.nan
    return pivot[months].reset_index()


def compute_global_month_avg(df, months):
    overall_avg = df["Price"].mean()
    monthly = {m: df.loc[df["month"] == m, "Price"].mean() for m in months}
    return {m: (monthly[m] if not pd.isna(monthly[m]) else overall_avg) for m in months}


def predict_future_months(nov, dec, jan, feb, global_avg_jan):
    known = [(1, nov), (2, dec), (3, jan), (4, feb)]
    known_vals = [
        (x, y) for x, y in known if y is not None and not (isinstance(y, float) and np.isnan(y))
    ]

    if len(known_vals) >= 2:
        xs = np.array([v[0] for v in known_vals])
        ys = np.array([v[1] for v in known_vals])
        coeffs = np.polyfit(xs, ys, 1)
        march = round(float(np.polyval(coeffs, 5)))
        april = round(float(np.polyval(coeffs, 6)))
    elif len(known_vals) == 1:
        march = round(known_vals[0][1])
        april = round(known_vals[0][1])
    else:
        march = round(global_avg_jan * 0.97)
        april = round(global_avg_jan * 0.95)

    return march, april


def cell_val(value, is_predicted):
    if is_predicted:
        return f"{int(round(value)):,}#prd"
    return int(round(value))


def fill_template(final_df, global_month_avg, template_path, output_path):
    if not template_path.exists():
        _create_fallback_template(template_path)

    wb = load_workbook(template_path)
    ws = wb.active

    actual_font = Font(name="Aptos Narrow", size=11)
    pred_font = Font(name="Aptos Narrow", size=11, color="0070C0")
    thin = Side(style="thin")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for i, row_data in final_df.iterrows():
        excel_row = DATA_START_ROW + i

        nov_raw = row_data.get("2025-11")
        dec_raw = row_data.get("2025-12")
        jan_raw = row_data.get("2026-01")
        feb_raw = row_data.get("2026-02")

        nov_actual = None if pd.isna(nov_raw) else round(nov_raw)
        dec_actual = None if pd.isna(dec_raw) else round(dec_raw)
        jan_actual = None if pd.isna(jan_raw) else round(jan_raw)
        feb_actual = None if pd.isna(feb_raw) else round(feb_raw)

        nov_is_pred = nov_actual is None
        dec_is_pred = dec_actual is None
        jan_is_pred = jan_actual is None
        feb_is_pred = feb_actual is None

        nov_val = nov_actual if not nov_is_pred else global_month_avg["2025-11"]
        dec_val = dec_actual if not dec_is_pred else global_month_avg["2025-12"]
        jan_val = jan_actual if not jan_is_pred else global_month_avg["2026-01"]
        feb_val = feb_actual if not feb_is_pred else global_month_avg["2026-02"]

        march_pred, april_pred = predict_future_months(
            nov_actual,
            dec_actual,
            jan_actual,
            feb_actual,
            global_avg_jan=global_month_avg["2026-01"],
        )

        year_val = int(row_data["Year"]) if not pd.isna(row_data["Year"]) else None
        avg_mileage_v = None if pd.isna(row_data["AvgMileage"]) else round(row_data["AvgMileage"])
        avg_price_v = None if pd.isna(row_data["AvgPrice"]) else round(row_data["AvgPrice"])
        avg_combined = (
            f"{avg_price_v:,} | {avg_mileage_v:,}"
            if avg_price_v is not None and avg_mileage_v is not None
            else None
        )

        row_values = [
            (row_data["Make"], False, None),
            (row_data["Model"], False, None),
            (year_val, False, "center"),
            (cell_val(nov_val, nov_is_pred), nov_is_pred, None),
            (cell_val(dec_val, dec_is_pred), dec_is_pred, None),
            (cell_val(jan_val, jan_is_pred), jan_is_pred, None),
            (cell_val(feb_val, feb_is_pred), feb_is_pred, None),
            (f"{march_pred:,}#prd", True, None),
            (f"{april_pred:,}#prd", True, None),
            (f"{april_pred:,}#prd", True, None),
            (avg_combined, False, None),
        ]

        for col_idx, (val, is_pred, align) in enumerate(row_values, start=1):
            cell = ws.cell(row=excel_row, column=col_idx, value=val)
            cell.font = pred_font if is_pred else actual_font
            cell.border = border
            if align:
                cell.alignment = Alignment(horizontal=align)
            if col_idx in (4, 5, 6, 7) and not is_pred:
                cell.number_format = "#,##0"

    col_widths = {
        "A": 18,
        "B": 22,
        "C": 10,
        "D": 16,
        "E": 16,
        "F": 16,
        "G": 16,
        "H": 18,
        "I": 28,
        "J": 18,
        "K": 24,
    }
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    wb.save(output_path)
    print(f"Saved -> {output_path}")


def main():
    print("Loading data...")
    df = load_and_prepare(CSV_PATH)

    print("Building monthly price pivot...")
    pivot = build_monthly_pivot(df, MONTHS)

    print("Computing global monthly averages...")
    global_month_avg = compute_global_month_avg(df, MONTHS)
    for m, avg in global_month_avg.items():
        print(f"  {m}: {avg:,.0f}")

    print("Aggregating avg mileage and overall avg price per vehicle...")
    mileage_col = _first_existing_column(df, ["Milleage", "Mileage"])
    if mileage_col is None:
        raise KeyError("Neither 'Milleage' nor 'Mileage' column exists in the dataset.")

    avg_mileage = (
        df.groupby(["Make", "Model", "Year"])[mileage_col]
        .mean()
        .reset_index()
        .rename(columns={mileage_col: "AvgMileage"})
    )
    avg_price_overall = (
        df.groupby(["Make", "Model", "Year"])["Price"]
        .mean()
        .reset_index()
        .rename(columns={"Price": "AvgPrice"})
    )

    final = pivot.merge(avg_mileage, on=["Make", "Model", "Year"], how="left")
    final = final.merge(avg_price_overall, on=["Make", "Model", "Year"], how="left")
    print(f"Total unique vehicles: {len(final)}")

    print("Writing to Excel template...")
    fill_template(final, global_month_avg, TEMPLATE_PATH, OUTPUT_PATH)
    print(f"\nDone! {len(final)} vehicles written to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Loading data...
Building monthly price pivot...
Computing global monthly averages...
  2025-11: 7,695,691
  2025-12: 7,422,176
  2026-01: 7,114,552
  2026-02: 6,875,422
Aggregating avg mileage and overall avg price per vehicle...
Total unique vehicles: 9904
Writing to Excel template...


PermissionError: [Errno 13] Permission denied: 'd:\\AutoInsight Dashboard\\autoinsightcs70\\model_training\\Template_for_Model_Filled.xlsx'